# L3 - LLM Data Cleaning
Piano di cleaning e narrativa audit per tutti i CSV Olist.
LLM genera: piano strutturato JSON + narrativa prima/dopo per ogni tabella.

In [ ]:
import os, json, re, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')

_HERE        = Path(os.path.abspath('__file__')).parent
_ROOT        = _HERE.parent
RAW_PATH     = str(_ROOT / '1_raw_data') + os.sep
CLEANED_PATH = str(_ROOT / '3_cleaned_data') + os.sep
OUT_PATH     = str(_HERE / 'outputs') + os.sep
os.makedirs(OUT_PATH, exist_ok=True)
print('RAW_PATH    :', RAW_PATH)
print('CLEANED_PATH:', CLEANED_PATH)
print('OUT_PATH    :', OUT_PATH)

In [ ]:
import requests

LLM_PROVIDER      = 'ollama'
OLLAMA_MODEL      = 'llama3.2:3b'
OLLAMA_URL        = 'http://localhost:11434/api/chat'
OPENAI_MODEL      = 'gpt-4o-mini'
ANTHROPIC_MODEL   = 'claude-haiku-4-5-20251001'
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

def call_llm(system_prompt, user_prompt, max_tokens=1500):
    try:
        if LLM_PROVIDER == 'ollama':
            r = requests.post(OLLAMA_URL, json={
                'model': OLLAMA_MODEL, 'stream': False,
                'messages': [{'role':'system','content':system_prompt},
                             {'role':'user','content':user_prompt}]
            }, timeout=120)
            r.raise_for_status()
            return r.json()['message']['content']
        elif LLM_PROVIDER == 'openai':
            import openai
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(model=OPENAI_MODEL,
                messages=[{'role':'system','content':system_prompt},
                           {'role':'user','content':user_prompt}])
            return resp.choices[0].message.content
        elif LLM_PROVIDER == 'anthropic':
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            msg = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                system=system_prompt,
                messages=[{'role':'user','content':user_prompt}])
            return msg.content[0].text
    except Exception as e:
        return f'[LLM ERROR] {e}'

print(f'Provider: {LLM_PROVIDER}')

In [ ]:
# Mappatura raw -> cleaned per ogni tabella
TABLES = {
    'orders':      ('olist_orders_dataset.csv',         'fact_orders.csv'),
    'customers':   ('olist_customers_dataset.csv',      'dim_customers.csv'),
    'order_items': ('olist_order_items_dataset.csv',    'fact_order_items.csv'),
    'products':    ('olist_products_dataset.csv',       'dim_products.csv'),
    'sellers':     ('olist_sellers_dataset.csv',        'dim_sellers.csv'),
    'payments':    ('olist_order_payments_dataset.csv', 'fact_payments.csv'),
    'reviews':     ('olist_order_reviews_dataset.csv',  'fact_reviews.csv'),
    'geolocation': ('olist_geolocation_dataset.csv',    'dim_geolocation.csv'),
}

raw_dfs     = {}
cleaned_dfs = {}

for name, (raw_file, clean_file) in TABLES.items():
    rp = RAW_PATH + raw_file
    cp = CLEANED_PATH + clean_file
    if os.path.exists(rp):
        raw_dfs[name] = pd.read_csv(rp)
    else:
        print(f'  [SKIP raw] {raw_file}')
        continue
    if os.path.exists(cp):
        cleaned_dfs[name] = pd.read_csv(cp)
        print(f'  [{name}] raw={len(raw_dfs[name]):,} -> cleaned={len(cleaned_dfs[name]):,} righe')
    else:
        print(f'  [{name}] raw={len(raw_dfs[name]):,} righe (nessun cleaned trovato)')

In [ ]:
# Carica traduzione categorie per products
cat_translation_file = RAW_PATH + 'product_category_name_translation.csv'
if os.path.exists(cat_translation_file):
    cat_tr = pd.read_csv(cat_translation_file, encoding='utf-8-sig')
    cat_map = dict(zip(cat_tr.iloc[:,0], cat_tr.iloc[:,1]))
    print(f'Traduzione categorie caricata: {len(cat_map)} mappings')
else:
    cat_map = {}
    print('File traduzione categorie non trovato')

# Carica audit log se esiste
audit_file = CLEANED_PATH + 'audit_log.csv'
if os.path.exists(audit_file):
    audit_log = pd.read_csv(audit_file)
    print(f'Audit log caricato: {len(audit_log)} righe')
else:
    audit_log = pd.DataFrame()
    print('Audit log non trovato')

In [ ]:
SYSTEM_PLAN = (
    'Sei un senior Data Engineer che costruisce pipeline ETL per un DW e-commerce (Olist, Brasile). '
    'Produci piani di cleaning strutturati in JSON, direttamente traducibili in codice pandas. '
    'Priorita per impatto sul business. Restituisci SOLO JSON valido. '
    'Scrivi le descrizioni degli step in italiano.'
)

SYSTEM_AUDIT = (
    'Sei un Data Quality Analyst che scrive narrative di audit per un DW e-commerce. '
    'Date statistiche prima/dopo cleaning, produci una narrativa professionale: '
    'cosa e stato fatto, perche, miglioramento misurabile. '
    'Scrivi in italiano. Pubblico: tecnico + business.'
)

print('Prompt caricati.')

In [ ]:
def build_dirty_profile(name, df):
    profile = {
        'table': name,
        'rows': len(df),
        'columns': list(df.columns),
        'missing': {c: int(df[c].isna().sum()) for c in df.columns if df[c].isna().any()},
        'duplicates': int(df.duplicated().sum()),
        'dtypes': {c: str(df[c].dtype) for c in df.columns}
    }
    # Aggiungi statistiche numeriche
    num_cols = df.select_dtypes(include='number').columns.tolist()
    if num_cols:
        profile['numeric_stats'] = {
            c: {'min': round(float(df[c].min()), 2),
                'max': round(float(df[c].max()), 2),
                'mean': round(float(df[c].mean()), 2)}
            for c in num_cols[:5]
        }
    if name == 'products' and 'product_category_name' in df.columns:
        profile['category_info'] = {
            'n_unique': int(df['product_category_name'].nunique()),
            'n_missing': int(df['product_category_name'].isna().sum()),
            'needs_translation': True
        }
    return profile


def build_before_after(name, raw_df, clean_df):
    ba = {
        'table': name,
        'before': {
            'rows': len(raw_df), 'cols': raw_df.shape[1],
            'missing_total': int(raw_df.isna().sum().sum()),
            'duplicates': int(raw_df.duplicated().sum())
        },
        'after': {
            'rows': len(clean_df), 'cols': clean_df.shape[1],
            'missing_total': int(clean_df.isna().sum().sum()),
            'duplicates': int(clean_df.duplicated().sum()),
            'new_columns': [c for c in clean_df.columns if c not in raw_df.columns]
        }
    }
    return ba

print('Funzioni pronte.')

In [ ]:
results = {}

for name, raw_df in raw_dfs.items():
    print(f'Processing: {name}...')
    clean_df = cleaned_dfs.get(name)

    # --- Piano di cleaning LLM ---
    profile = build_dirty_profile(name, raw_df)
    plan_prompt = (
        f'Produci un piano di cleaning per la tabella Olist "{name}".\n\n'
        f'Profilo dati:\n{json.dumps(profile, indent=2, ensure_ascii=False)}\n\n'
        f'Restituisci JSON con struttura:\n'
        '{"table": "nome", "steps": [{"step_id": 1, "name": "", '
        '"description": "", "columns": [], "pandas_method": "", '
        '"priority": "high|medium|low", "business_impact": ""}]}'
    )
    plan_resp = call_llm(SYSTEM_PLAN, plan_prompt, max_tokens=1500)
    try:
        m = re.search(r'```(?:json)?\n?(.+?)```', plan_resp, re.DOTALL)
        plan = json.loads(m.group(1).strip() if m else plan_resp.strip())
    except:
        plan = {'table': name, 'steps': [], 'raw': plan_resp}

    # --- Narrativa audit LLM ---
    if clean_df is not None:
        ba = build_before_after(name, raw_df, clean_df)
        audit_sample = audit_log[audit_log['table'] == name].head(5).to_dict('records') if len(audit_log) > 0 and 'table' in audit_log.columns else []
        audit_prompt = (
            f'Narrativa audit per tabella Olist "{name}".\n\n'
            f'Statistiche prima/dopo:\n{json.dumps(ba, indent=2, ensure_ascii=False)}\n\n'
            f'Campione audit log:\n{json.dumps(audit_sample, indent=2, ensure_ascii=False, default=str)}\n\n'
            f'Includi: sintesi intervento, trasformazioni principali, impatto qualita, limitazioni, raccomandazioni.'
        )
        narrative = call_llm(SYSTEM_AUDIT, audit_prompt, max_tokens=1200)
    else:
        ba = None
        narrative = 'Nessuna versione cleaned disponibile per questa tabella.'

    results[name] = {'plan': plan, 'narrative': narrative, 'before_after': ba}

    # --- Display inline ---
    display(Markdown(f'---\n### {name.upper()}'))
    display(Markdown('**Piano di Cleaning**\n\n' + plan_resp))
    display(Markdown('**Narrativa Audit**\n\n' + narrative))

    # --- Salva file ---
    out_md = OUT_PATH + f'L3_{name}_cleaning.md'
    out_json = OUT_PATH + f'L3_{name}_plan.json'
    with open(out_md, 'w', encoding='utf-8') as f:
        f.write(f'# L3 - Cleaning: {name}\n\n')
        f.write(f'**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M")} | **Provider:** {LLM_PROVIDER}\n\n')
        f.write('## Piano di Cleaning\n\n' + plan_resp + '\n\n')
        f.write('## Narrativa Audit\n\n' + narrative + '\n')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(plan, f, indent=2, ensure_ascii=False)
    print(f'  Salvato -> {out_md}')

print(f'\nCleaning completato per {len(results)} tabelle.')

In [ ]:
# Riepilogo aggregato
agg_file = OUT_PATH + 'L3_RIEPILOGO.md'
with open(agg_file, 'w', encoding='utf-8') as f:
    f.write('# L3 - Riepilogo Cleaning Plans\n\n')
    f.write('| Tabella | Righe raw | Righe cleaned | Step piano | Nuove colonne |\n')
    f.write('|---------|-----------|---------------|------------|---------------|\n')
    for name, res in results.items():
        raw_rows = len(raw_dfs[name])
        clean_rows = len(cleaned_dfs[name]) if name in cleaned_dfs else '-'
        n_steps = len(res['plan'].get('steps', []))
        new_cols = len(res['before_after']['after']['new_columns']) if res['before_after'] else '-'
        f.write(f'| {name} | {raw_rows:,} | {clean_rows} | {n_steps} | {new_cols} |\n')
print(f'Riepilogo salvato: {agg_file}')